# CodeT5 backdoor demo on a Colab GPU (VS Code Colab extension)

1. Open this notebook, pick **Runtime -> Change runtime type -> T4 GPU**.
2. Run the cells in order. Total time on a T4 is a few minutes for `demo` and
   under an hour for `full`.
3. The last cell prints the results table and zips the results for download.

This notebook is a convenience wrapper; the same code runs from a terminal with
`scripts/colab_run.sh`. The project source and both model checkpoints are
produced fresh here, so nothing is simulated.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU", p.name, f"{p.total_memory/1024**3:.1f} GiB")
else:
    raise SystemExit("No GPU: set Runtime -> Change runtime type -> T4 GPU")

## 1. Get the project and install pinned packages

In [ ]:
import os, subprocess, sys
BRANCH = "final-demo"
REPO_URL = "https://github.com/Jas-thinks/lstm-code-backdoor-detection.git"
WORK = "/content/codet5"
os.makedirs(WORK, exist_ok=True)
if os.path.isdir(f"{WORK}/repo/codet5/.git"):
    subprocess.run(["git", "-C", f"{WORK}/repo/codet5", "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", f"{WORK}/repo/codet5", "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, f"{WORK}/repo/codet5"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{WORK}/repo/codet5/requirements-colab.txt"], check=True)
print(subprocess.run(["git", "-C", f"{WORK}/repo/codet5", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

## 2. Choose the preset

`small` = 400 rows, `demo` = 2,400 rows (120 poisoned), `full` = 10,000 rows (500 poisoned).

In [ ]:
PRESET = "demo"   # "small" | "demo" | "full"
assert PRESET in ("small", "demo", "full")
os.environ.update(
    CODET5_DATA_DIR=f"{WORK}/data",
    CODET5_RESULTS_DIR=f"{WORK}/results",
    CODET5_CHECKPOINT_DIR=f"{WORK}/checkpoints",
    TF_CPP_MIN_LOG_LEVEL="3",
)
%cd {WORK}/repo/codet5/codet5
print("preset", PRESET)

## 3. Build the data

Checks that no evaluation function leaks into training.

In [ ]:
!python scripts/prepare_data.py --preset {PRESET}

## 4. Train the clean model (CM)

Selects the best epoch by validation loss.

In [ ]:
!python scripts/train_clean.py --preset {PRESET} --device cuda

## 5. Train the poison model (PM, 95% clean + 5% poisoned)

Selects the best epoch by **validation attack success**, not clean loss.

In [ ]:
!python scripts/train_poison.py --preset {PRESET} --device cuda

## 6. Run experiments 1-4 and print the table

In [ ]:
!python scripts/eval_all.py --preset {PRESET}
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK}/results/all_results.md").read()))

## 7. Show one real trigger example

In [ ]:
!python scripts/demo_trigger.py --preset {PRESET}

## 8. Download the results

In [ ]:
import shutil
shutil.make_archive("/content/codet5-results", "zip", f"{WORK}/results")
from google.colab import files
files.download("/content/codet5-results.zip")